[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Why Pandas &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Print Tromso's mean for the year, rounded to two places, by filtering `readings` on the station's name.

In [2]:
tromso = readings[readings["station"] == "Tromso"]

print(round(tromso["celsius"].mean(), 2))

3.5


The mask keeps Tromso's 8,760 rows, and the mean of their `celsius` column is the formula's 3.5.

**2.** Show the coldest hour at each station, with its time, using `idxmin`.

In [3]:
coldest = readings.groupby("station")["celsius"].idxmin()
readings.loc[coldest]

,station,time,celsius
1836,Bergen,2025-01-20 03:00:00,-4.8
1453,Oslo,2025-01-16 03:00:00,-6.3
1070,Svalbard,2025-01-12 03:00:00,-17.3
687,Tromso,2025-01-08 03:00:00,-9.2


`idxmin` is `idxmax`'s counterpart. Every station's coldest hour is at 03:00 in January, the bottom of both of the formula's swings.

**3.** Print each station's number of readings, with `readings.groupby("station")["celsius"].count()`, and say which station has fewer and why.

In [4]:
readings.groupby("station")["celsius"].count()

station
Bergen      8760
Oslo        8760
Svalbard    8736
Tromso      8760
Name: celsius, dtype: int64

`count` counts the values that are not missing. Svalbard has 8,736, 24 fewer than the others, because it sent nothing on 2 March.

**4.** From `monthly`, print each station's coldest monthly mean, with `monthly.min(axis=1)`.

In [5]:
monthly = readings.groupby(["station", readings["time"].dt.month])["celsius"].mean().unstack().round(1)

monthly.min(axis=1)

station
Bergen      -0.9
Oslo        -2.4
Svalbard   -13.4
Tromso      -5.4
dtype: float64

The first line builds `monthly` as the worked example did, since this notebook runs on its own. `axis=1` takes the minimum across the columns, one for each row, so the answer has one value per station. Every station's coldest month is January.

**5.** Make a table with each station's warmest reading of the year beside its latitude, the way the last worked example lined up the means.

In [6]:
highest = readings.groupby("station")["celsius"].max()
latitude = stations.set_index("station")["latitude"]

pd.DataFrame({"highest": highest, "latitude": latitude})

,highest,latitude
station,,
Bergen,20.8,60.39
Kirkenes,NaN,69.73
Oslo,19.3,59.91
Svalbard,8.3,78.22
Tromso,16.2,69.65


The two Series line up by station name. Kirkenes has a latitude and no readings, so its highest reading is `NaN`.

---

&#8592; **Back to:** [Why Pandas](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/01-why-pandas.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
